# Embedding Cluster Spike

Can unsupervised clustering on variable embeddings discover the concept hierarchy?

1. Embed all 438K variables with `all-MiniLM-L6-v2`
2. PCA → neighbors → Leiden clustering
3. Compare clusters to hand-curated concept assignments

In [ ]:
import json
from pathlib import Path
import numpy as np

# Load all variables
llm_dir = Path("output/llm-concepts-v4")
variables = []

for path in sorted(llm_dir.glob("phs*.json")):
    with open(path) as f:
        data = json.load(f)
    for table in data.get("tables", []):
        for var in table.get("variables", []):
            variables.append({
                "name": var.get("name", ""),
                "description": var.get("description", ""),
                "concept_id": var.get("concept_id", ""),
                "study_id": data.get("studyId"),
            })

print(f"Loaded {len(variables):,} variables")
print(f"Classified: {sum(1 for v in variables if v['concept_id']):,}")
print(f"Unclassified: {sum(1 for v in variables if not v['concept_id']):,}")

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="mps")

# Embed: "VARIABLE_NAME: DESCRIPTION"
texts = [f"{v['name']}: {v['description']}" for v in variables]
print(f"Embedding {len(texts):,} texts on {model.device}...")
embeddings = model.encode(texts, batch_size=512, show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
import scanpy as sc
import anndata as ad

# Build AnnData object (scanpy's standard container)
adata = ad.AnnData(X=embeddings)
adata.obs["name"] = [v["name"] for v in variables]
adata.obs["description"] = [v["description"] for v in variables]
adata.obs["concept_id"] = [v["concept_id"] if v["concept_id"] else "unclassified" for v in variables]
adata.obs["study_id"] = [v["study_id"] for v in variables]

# Extract top-level concept (namespace:name -> just the name part, or the ncpi parent)
# For now, use the concept_id as-is for coloring
print(adata)
print(f"Unique concepts: {adata.obs['concept_id'].nunique()}")

In [ ]:
# Standard scanpy workflow: PCA -> neighbors -> UMAP -> Leiden
print("PCA...")
sc.tl.pca(adata, n_comps=50)

print("Computing neighbors...")
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50)

print("UMAP...")
sc.tl.umap(adata)

print("Leiden clustering (coarse, resolution=0.5)...")
sc.tl.leiden(adata, resolution=0.5, key_added="leiden_coarse")

print("Leiden clustering (fine, resolution=2.0)...")
sc.tl.leiden(adata, resolution=2.0, key_added="leiden_fine")

print(f"Coarse clusters: {adata.obs['leiden_coarse'].nunique()}")
print(f"Fine clusters: {adata.obs['leiden_fine'].nunique()}")

In [ ]:
# Visualize: UMAP colored by coarse Leiden clusters
sc.pl.umap(adata, color="leiden_coarse", legend_loc="on data", 
           title="Coarse Leiden clusters (res=0.5)", frameon=False)

In [ ]:
# Compare: for each coarse cluster, what are the top concept_ids?
import pandas as pd

df = adata.obs[["leiden_coarse", "concept_id"]].copy()
classified = df[df["concept_id"] != "unclassified"]

for cluster in sorted(df["leiden_coarse"].unique(), key=int):
    cluster_df = classified[classified["leiden_coarse"] == cluster]
    total_in_cluster = len(df[df["leiden_coarse"] == cluster])
    if len(cluster_df) == 0:
        print(f"\nCluster {cluster} ({total_in_cluster:,} vars): all unclassified")
        continue
    top = cluster_df["concept_id"].value_counts().head(5)
    purity = top.iloc[0] / len(cluster_df) * 100
    print(f"\nCluster {cluster} ({total_in_cluster:,} vars, {len(cluster_df):,} classified, purity={purity:.0f}%):")
    for concept, count in top.items():
        print(f"  {concept}: {count:,}")

In [ ]:
# Visualize: UMAP colored by existing concept_id (top 20 + "other")
top_concepts = adata.obs["concept_id"].value_counts().head(20).index.tolist()
adata.obs["concept_top20"] = adata.obs["concept_id"].where(
    adata.obs["concept_id"].isin(top_concepts), other="other"
)
sc.pl.umap(adata, color="concept_top20",
           title="Existing concept assignments (top 20)", frameon=False)

In [ ]:
# Key question: how well do Leiden clusters align with existing concepts?
# Adjusted Rand Index and Normalized Mutual Information
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Only compare on classified variables
mask = adata.obs["concept_id"] != "unclassified"
concepts = adata.obs.loc[mask, "concept_id"]
coarse = adata.obs.loc[mask, "leiden_coarse"]
fine = adata.obs.loc[mask, "leiden_fine"]

print(f"Comparing on {mask.sum():,} classified variables")
print(f"\nCoarse Leiden vs concepts:")
print(f"  ARI:  {adjusted_rand_score(concepts, coarse):.3f}")
print(f"  NMI:  {normalized_mutual_info_score(concepts, coarse):.3f}")
print(f"\nFine Leiden vs concepts:")
print(f"  ARI:  {adjusted_rand_score(concepts, fine):.3f}")
print(f"  NMI:  {normalized_mutual_info_score(concepts, fine):.3f}")

In [ ]:
# Where do unclassified variables land?
unclassified = df[df["concept_id"] == "unclassified"]
print(f"Unclassified variables by cluster:\n")
for cluster in sorted(unclassified["leiden_coarse"].unique(), key=int):
    n_unclass = len(unclassified[unclassified["leiden_coarse"] == cluster])
    # What's the dominant concept in this cluster?
    cluster_classified = classified[classified["leiden_coarse"] == cluster]
    if len(cluster_classified) > 0:
        dominant = cluster_classified["concept_id"].value_counts().index[0]
        dominant_pct = cluster_classified["concept_id"].value_counts().iloc[0] / len(cluster_classified) * 100
    else:
        dominant = "(none)"
        dominant_pct = 0
    print(f"  Cluster {cluster}: {n_unclass:,} unclassified — dominant concept: {dominant} ({dominant_pct:.0f}%)")